## Installation
Install the decent-bench package and PyTorch if not already present.

In [ ]:
%pip install -e ../../.
%pip install torch torchvision --index-url https://download.pytorch.org/whl/cu128

## Imports

In [ ]:
from pathlib import Path

import networkx as nx
import numpy as np
import torch
from torch import nn
from torchvision import transforms
from torchvision.datasets import MNIST

import decent_bench.algorithms.p2p as dec_algorithms
import decent_bench.utils.interoperability as iop
from decent_bench import benchmark
from decent_bench.agents import Agent
from decent_bench.algorithms.utils import pytorch_initialization
from decent_bench.costs import PyTorchCost
from decent_bench.datasets import PyTorchDatasetHandler
from decent_bench.metrics import metric_library as ml
from decent_bench.networks import P2PNetwork
from decent_bench.schemes import GaussianNoise, TopK, UniformActivationRate, UniformDropRate
from decent_bench.utils.checkpoint_manager import CheckpointManager
from decent_bench.utils.pytorch_utils import ArgmaxActivation, SimpleLinearModel
from decent_bench.utils.types import SupportedDevices

## Experiment parameters

In [ ]:
checkpoint_path = Path("results/mnist")
iterations = 1000
n_trials = 5
state_snapshot_period = 50
n_agents = 5
n_neighbors = 4
samples_per_partition = 1000
heterogeneity = False
targets_per_partition = 2  # If heterogeneity is True, this is the number of classes assigned to each partition
batch_size = 32
device = SupportedDevices.CPU

# Define network impairments, set to None to disable
activation = UniformActivationRate(0.8)
noise = GaussianNoise(0.0, 0.01)
compression = TopK(0.1)
drops = UniformDropRate(0.2)

# Set seed
iop.set_seed(47)

## Define which metrics to calculate

In [ ]:
table_metrics = [
    ml.ConsensusError([min, np.average, max]),
    ml.GradientCalls([np.average, sum]),
    ml.SentMessages([np.average, sum]),
    ml.Accuracy([min, np.average, max], fmt=".2%"),
    ml.Precision([min, np.average, max], fmt=".2%"),
    ml.Recall([min, np.average, max], fmt=".2%"),
    ml.Loss([min, np.average, max]),
]

plot_metrics = [
    ml.ConsensusError([], x_log=False, y_log=True),
    ml.Accuracy([], x_log=False, y_log=False),
    ml.Precision([], x_log=False, y_log=False),
    ml.Recall([], x_log=False, y_log=False),
    ml.Loss([], x_log=False, y_log=False),
]

## Define PyTorch model generator

In [ ]:
def model_generator() -> torch.nn.Module:
    """Generate a simple linear model for the MNIST dataset."""
    return SimpleLinearModel(
        input_size=28 * 28,
        hidden_sizes=[32, 16],
        output_size=10,
    )

## Create MNIST datasets

In [ ]:
transform = transforms.Compose([transforms.ToTensor(), transforms.Lambda(lambda x: x.view(-1))])
mnist_train = MNIST(
    root="data",
    train=True,
    download=True,
    transform=transform,
    target_transform=torch.tensor,
)
mnist_test = MNIST(
    root="data",
    train=False,
    download=True,
    transform=transform,
    target_transform=torch.tensor,
)
train_dataset = PyTorchDatasetHandler(
    torch_dataset=mnist_train,
    n_features=28 * 28,
    n_targets=10,
    n_partitions=n_agents,
    samples_per_partition=samples_per_partition,
    heterogeneity=heterogeneity,
    targets_per_partition=targets_per_partition,
)
test_dataset = PyTorchDatasetHandler(
    torch_dataset=mnist_test,
    n_features=28 * 28,
    n_targets=10,
    n_partitions=n_agents,
    heterogeneity=heterogeneity,
    targets_per_partition=targets_per_partition,
)

## Create checkpoint manager

In [ ]:
cm = CheckpointManager(
    checkpoint_dir=checkpoint_path,
    checkpoint_step=None,
    keep_n_checkpoints=1,
    benchmark_metadata={
        "dataset": "MNIST",
        "n_agents": n_agents,
        "n_neighbors": n_neighbors,
        "heterogeneity": heterogeneity,
        "targets_per_partition": targets_per_partition,
        "drops": drops.__class__.__name__ if drops else None,
        "activity": activation.__class__.__name__ if activation else None,
        "compression": compression.__class__.__name__ if compression else None,
        "noise": noise.__class__.__name__ if noise else None,
    },
)

## Create costs, agents, network, and benchmark problem

In [ ]:
# Each cost function gets a different partition of the dataset, but all use the same model architecture.
# Final activation is set to argmax since this is a classification problem, but can be set to None for regression tasks.
# The loss function is cross-entropy, which is standard for classification tasks, but can be changed as needed for
# different types of problems. The batch size and device are also specified here, but can be adjusted based on the
# size of the dataset and available hardware.
costs = [
    PyTorchCost(
        dataset=p,
        model=model_generator(),
        loss_fn=nn.CrossEntropyLoss(),
        final_activation=ArgmaxActivation(),
        batch_size=batch_size,
        device=device,
    )
    for p in train_dataset.get_partitions()
]
agents = [
    Agent(
        i,
        cost,
        state_snapshot_period=state_snapshot_period,
        activation=activation,
    )
    for i, cost in enumerate(costs)
]
graph = nx.random_regular_graph(d=n_neighbors, n=n_agents, seed=iop.get_seed())
network = P2PNetwork(
    graph=graph,
    agents=agents,
    message_noise=noise,
    message_compression=compression,
    message_drop=drops,
)
problem = benchmark.BenchmarkProblem(
    network=network,
    test_data=test_dataset.get_datapoints(),
)

## Create algorithms

In [ ]:
x0 = pytorch_initialization(network, all_same=True)
algorithms = [
    dec_algorithms.DGD(
        step_size=0.1,
        iterations=iterations,
        x0=x0,
    ),
    dec_algorithms.KGT(
        iterations=iterations,
        num_local_steps=10,
        step_size=0.01,
        aux_step_size=0.5,
        x0=x0,
    ),
    dec_algorithms.ProxSkip(
        iterations=iterations,
        step_size=0.1,
        aux_step_size=0.1,
        comm_probability=0.2,
        chi=1.0,
        x0=x0,
    ),
    dec_algorithms.LED(
        iterations=iterations,
        num_local_steps=10,
        step_size=0.01,
        aux_step_size=0.01,
        x0=x0,
    ),
    dec_algorithms.LT_ADMM(
        iterations=iterations,
        num_local_steps=10,
        step_size=0.01,
        aux_step_size=0.01,
        penalty=1.0,
        x0=x0,
    ),
    dec_algorithms.LT_ADMM_VR(
        iterations=iterations,
        num_local_steps=10,
        step_size=0.01,
        aux_step_size=0.01,
        penalty=1.0,
        x0=x0,
    ),
]

## Start benchmark execution

In [ ]:
result = benchmark.benchmark(
    algorithms=algorithms,
    benchmark_problem=problem,
    n_trials=n_trials,
    show_speed=True,
    show_trial=True,
    checkpoint_manager=cm,
)

metric_result = benchmark.compute_metrics(
    benchmark_result=result,
    checkpoint_manager=cm,
    table_metrics=table_metrics,
    plot_metrics=plot_metrics,
)

benchmark.display_metrics(
    metrics_result=metric_result,
    checkpoint_manager=cm,
)

## Optionally resume benchmark from checkpoint
Uncomment the code in the code block below to resume an interrupted benchmark execution

In [ ]:
# result = benchmark.resume_benchmark(
#     checkpoint_manager=cm,
#     create_backup=False,
#     show_speed=True,
#     show_trial=True,
# )

# metric_result = benchmark.compute_metrics(
#     benchmark_result=result,
#     checkpoint_manager=cm,
#     table_metrics=table_metrics,
#     plot_metrics=plot_metrics,
# )

# benchmark.display_metrics(
#     metrics_result=metric_result,
#     checkpoint_manager=cm,
# )